---
title: "One Analysis Table from Several Sources"
subtitle: "Joining the HarborLink extracts"
jupyter: data-extraction-lesson
---

In [ ]:
from pathlib import Path
import os

if not Path("data/raw/analysis/customers.csv").exists():
    os.chdir(Path("..").resolve())

print("Working directory:", Path.cwd())

## Learning objectives

This section will later use **matplotlib** and **seaborn** to explore HarborLink data. Before any chart is useful, the extracts from the previous section must become **one analysis table**.

By the end of this session you should be able to:

1. Explain why a collection of DataFrames is not yet an analysis dataset.
2. Choose an analytical unit and the keys that connect sources.
3. Load the larger HarborLink extracts (CSV, Excel, JSON, SQLite, API).
4. Explain, for each source, which key attaches which columns to an invoice row.
5. Aggregate a one-to-many table before joining it.
6. Build the invoice-level DataFrame step by step with `pandas.merge`.
7. Add a few analysis columns, save the table, and forward each source's variable descriptions into Excel.

We do **not** start the full visualisation lesson in this chapter. The deliverable is the analysis DataFrame.

## Why this step comes first

The previous section showed that HarborLink information lives in several places: a customer export, an invoice workbook, a CRM JSON file, a finance database, an API snapshot. Each load produced a DataFrame. Those frames are not yet something we can plot as “the HarborLink dataset”.

Exploratory analysis needs:

- a **defined row** (what is one observation?);
- **variables on that same row**;
- **keys** that make the join legitimate;
- a table large enough for patterns to appear, but small enough to run live.

The eight-customer extract from the previous section was perfect for teaching `read_csv` and `json_normalize`. It is too small for distributions and comparisons. We therefore keep the same fictitious company and the same people — Elena Vargas is still customer 1827 — and work with a **panel extract**: 240 customers and their monthly invoices.

## The analytical unit is the invoice

The late-payment question is not “one row per customer” and not “one row per email”. The useful statistical unit is:

> **One row per invoice** — one billing cycle for one customer.

That choice decides everything that follows. We **start** from the invoice table. Every other table is used only to **add columns** to those invoice rows. We never replace the invoice as the row.

If we later need one row per customer, we can aggregate this table. Starting too aggregated would throw away the monthly variation that plots will show.

## What a merge actually does

A merge does not “blend two files together”. It walks along the invoice table, row by row, and **looks up extra fields** in another table.

Take Elena Vargas’s September invoice. After loading, that row already knows the invoice identifier, the customer identifier, the dates, the amount and the finance status. It does **not** yet know her name, her country, her plan, whether a payment arrived, or whether she emailed support.

Those facts live in other tables. They can be copied onto her invoice row only if we know **which field is the shared key**.

```{mermaid}
flowchart TD
    I["Invoice INV-2026-0901<br/>customer_id = 1827<br/>subscription_id = SUB-1827<br/>period = 2026-09<br/>amount = 49.90"]
    I -->|customer_id| C[Customer attributes]
    I -->|subscription_id| S[Plan and monthly amount]
    I -->|invoice_id| P[Payment date and payment status]
    I -->|customer_id and period| R[CRM counts for that month]
    I -->|invoice_id| A[September API status]
```

The picture is always the same:

1. Keep the invoice row.
2. Name the key (or keys).
3. Find the matching row(s) in the other table.
4. Copy selected columns onto the invoice.
5. If several rows match, stop and aggregate first. Otherwise the invoice would be duplicated.

`how="left"` means: **keep every invoice**, even when the other table has no match. The new columns are then missing (`NaN`). That is the correct behaviour for a historical invoice that does not appear in the September API.

## Load each source

The panel files live in `data/raw/analysis/`. 
<!--
They were generated with `python/generate_analysis_data.py` and include the customers from the previous section.

That script does not write the extracts alone. In the same run it calls `write_source_description_files`, which places a companion CSV next to each source (`customers_variables.csv`, `invoices_variables.csv`, `subscriptions_variables.csv`, `payments_variables.csv`, `interactions_variables.csv`, `api_invoices_variables.csv`). SQLite does not store those sentences inside `company.sqlite`; they sit next to the database, one CSV per table that we will join. Every companion has three columns: `variable`, `type`, `definition`. They are not the data. They are the meaning of the columns, stored **when the files are constructed** so that a later merge can copy the sentences with the columns.
-->

In [ ]:
import json
import sqlite3

import pandas as pd
import requests

customers = pd.read_csv(
    "data/raw/analysis/customers.csv",
    parse_dates=["signup_date"],
)
invoices = pd.read_excel(
    "data/raw/analysis/invoices.xlsx",
    sheet_name="invoices",
    parse_dates=["invoice_date", "due_date"],
)

connection = sqlite3.connect("data/raw/analysis/company.sqlite")
subscriptions = pd.read_sql_query("SELECT * FROM subscriptions", connection)
payments = pd.read_sql_query(
    """
    SELECT payment_id, invoice_id, payment_date, status AS payment_status
    FROM payments
    """,
    connection,
    parse_dates=["payment_date"],
)
connection.close()

with open("data/raw/analysis/interactions.json", encoding="utf-8") as file:
    interaction_documents = json.load(file)

contacts = pd.json_normalize(
    interaction_documents,
    record_path="contacts",
    meta=["customer_id"],
)
contacts["date"] = pd.to_datetime(contacts["date"])
contacts["period"] = contacts["date"].dt.strftime("%Y-%m")

In [ ]:
from python.mock_api import start_background_server

API_URL = start_background_server() + "/panel/invoices"
try:
    response = requests.get(API_URL, timeout=2)
    response.raise_for_status()
    api_payload = response.json()
except (requests.RequestException, OSError):
    with open("data/raw/analysis/api_invoices.json", encoding="utf-8") as file:
        api_payload = json.load(file)

api_invoices = pd.DataFrame(api_payload["invoices"])

We now have five tables in memory. They are still separate. Nothing has been joined yet.

In [ ]:
for name, frame in [
    ("invoices", invoices),
    ("customers", customers),
    ("subscriptions", subscriptions),
    ("payments", payments),
    ("contacts", contacts),
    ("api_invoices", api_invoices),
]:
    print(f"{name:15} {frame.shape[0]:5} rows  {frame.shape[1]:2} columns")

`invoices` has one row per billing cycle. That is our starting table. The other frames exist only to enrich those rows.

We will follow **one invoice** through every join: `INV-2026-0901`, Elena Vargas, September 2026. After each merge we look at that same row again, to see which columns have appeared.

In [ ]:
TARGET_INVOICE = "INV-2026-0901"


def show_invoice(frame, columns):
    """Return the Elena September invoice, with only the columns we are studying."""
    row = frame.loc[frame["invoice_id"] == TARGET_INVOICE, columns]
    return row

- `TARGET_INVOICE` is just a string we reuse, so we always look at the same row.
- `show_invoice` filters `frame` to that identifier and keeps the listed columns. It does not change the data.

## Step 0 — Start from the invoice row

In [ ]:
analysis = invoices.copy()

show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_id",
        "subscription_id",
        "period",
        "invoice_date",
        "due_date",
        "amount",
        "status",
    ],
)

analysis.head()

`analysis = invoices.copy()` creates the working table as a copy of the invoice sheet. From now on, **the number of rows should stay equal to the number of invoices**, except if we make a mistake.

This row already contains three keys we will use:

- `customer_id` `1827` — to find Elena in the customer table;
- `subscription_id` `SUB-1827` — to find her plan;
- `invoice_id` `INV-2026-0901` — to find the payment and the API snapshot;
- `period` `2026-09` — together with `customer_id`, to find her CRM activity in September.

It does not yet contain her name, her country, her plan, a payment date, or any email count.

## Step 1 — Attach the customer to each invoice

Look at the customer table for the same identifier.

In [ ]:
customers.loc[
    customers["customer_id"] == 1827,
    ["customer_id", "customer_name", "company_name", "customer_segment", "country", "signup_date"],
]

There is **one** customer row for `1827`. Many invoices can belong to Elena (one per month), but she appears only once in `customers`. The relationship is **many invoices to one customer**.

The merge copies those customer columns onto every invoice that has `customer_id == 1827`.

In [ ]:
analysis = analysis.merge(
    customers,
    on="customer_id",
    how="left",
    validate="m:1",
)

Read the arguments one by one:

- `analysis` is the **left** table. Its rows are the invoices. They are the rows we keep.
- `customers` is the **right** table. We take columns from it.
- `on="customer_id"` is the shared key. pandas lines up left and right rows that have the same `customer_id`.
- `how="left"` keeps every invoice. If a `customer_id` were missing from `customers`, the new columns would be empty, but the invoice would still be there.
- `validate="m:1"` is a safety check: many left rows may share a key, but each key may appear **at most once** on the right. If Elena appeared twice in `customers`, pandas would raise an error instead of silently duplicating her invoices.

In [ ]:
show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_id",
        "amount",
        "status",
        "customer_name",
        "company_name",
        "customer_segment",
        "country",
    ],
)

The invoice is still one row. The amount is still 49.90. Four new fields have been copied from the customer table: name, company, segment and country. That is all a customer merge does.

## Step 2 — Attach the subscription to each invoice

The invoice already carries `subscription_id`. The subscription table stores the plan type and the monthly amount.

In [ ]:
subscriptions.loc[
    subscriptions["subscription_id"] == "SUB-1827",
    ["subscription_id", "customer_id", "subscription_type", "monthly_amount", "start_date"],
]

Again, one subscription row. Several invoices share that identifier. We do not need every subscription column: `customer_id` is already on the invoice. We select only the fields we want to add.

In [ ]:
analysis = analysis.merge(
    subscriptions[
        ["subscription_id", "subscription_type", "monthly_amount", "start_date"]
    ],
    on="subscription_id",
    how="left",
    validate="m:1",
)

- The key is now `subscription_id`, not `customer_id`.
- The list inside `subscriptions[...]` is **column selection**. It happens *before* the merge, so we do not import a second `customer_id` that would collide with the one we already have.
- `validate="m:1"` again: many invoices, one subscription.

In [ ]:
show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_name",
        "subscription_id",
        "subscription_type",
        "monthly_amount",
        "amount",
        "status",
    ],
)

Elena’s September row now also knows that the plan is premium and that the catalogue price is 49.90. `amount` (from the invoice) and `monthly_amount` (from the subscription) can later be compared. They are not the same column: one is this month’s billed amount, the other is the plan price.

## Step 3 — Attach the payment to each invoice

Payments are stored at the same grain as invoices: one payment record per `invoice_id`. The key is therefore the invoice identifier itself.

In [ ]:
payments.loc[
    payments["invoice_id"] == TARGET_INVOICE,
    ["payment_id", "invoice_id", "payment_date", "payment_status"],
]

This is a **one-to-one** relationship. Each invoice should match exactly one payment row. For an unpaid invoice the payment row still exists: `payment_date` is missing and `payment_status` is `unpaid`. A missing date is not a failed join. It is the business fact “no payment has been recorded”.

In [ ]:
analysis = analysis.merge(
    payments,
    on="invoice_id",
    how="left",
    validate="1:1",
)

- `on="invoice_id"` matches the invoice to its payment record.
- `validate="1:1"` is stricter than `m:1`. Each key may appear once on the left and once on the right. Two payment rows for the same invoice would be an error in our data model.

In [ ]:
show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_name",
        "amount",
        "status",
        "payment_id",
        "payment_date",
        "payment_status",
    ],
)

The row is still unique. `payment_date` is empty for Elena in September, and `payment_status` is `unpaid`. Finance status (`status`) and payment status agree here; we kept both so that a later check can compare them.

## Step 4 — Attach CRM activity, after collapsing several contacts to one row

Customer interactions are different. The CRM file is **one row per contact**, not one row per invoice. Elena can have several September messages.

In [ ]:
elena_contacts = contacts.loc[
    (contacts["customer_id"] == 1827) & (contacts["period"] == "2026-09"),
    ["customer_id", "period", "date", "channel", "interaction_type", "text"],
]
elena_contacts

If we merged these contacts directly onto her invoice, pandas would repeat the invoice once per contact. The amount 49.90 would appear several times. That would break every later count and every later plot.

We can see the mistake on this single invoice:

In [ ]:
show_invoice(analysis, ["invoice_id", "customer_id", "amount", "status"]).merge(
    elena_contacts,
    on="customer_id",
    how="left",
)

One invoice has become several rows. That is why a raw contact join is forbidden here.

The remedy is to **summarise first**. For each customer and each billing month we compute numbers: how many contacts, how many reminders, how many payment issues. After that, there is again **one right-hand row per invoice month**.

In [ ]:
contact_summary = contacts.groupby(["customer_id", "period"], as_index=False).agg(
    n_interactions=("interaction_type", "size"),
    n_reminders=("interaction_type", lambda series: int((series == "reminder").sum())),
    n_complaints=("interaction_type", lambda series: int((series == "complaint").sum())),
    n_payment_issues=(
        "interaction_type",
        lambda series: int((series == "payment_issue").sum()),
    ),
    n_emails=("channel", lambda series: int((series == "email").sum())),
)

contact_summary.loc[
    (contact_summary["customer_id"] == 1827) & (contact_summary["period"] == "2026-09")
]

- `groupby(["customer_id", "period"])` splits the contacts into groups that have the same customer and the same month.
- `as_index=False` keeps those two keys as ordinary columns, so they can be used in `merge`.
- `agg(...)` replaces the group by a few numbers. `size` counts the rows. The `lambda` expressions count how many values equal `"reminder"`, `"complaint"`, and so on.

Elena’s September emails are now a single summary row. We can attach it to her invoice with **two keys**: the customer and the month. Using only `customer_id` would pour all of her historical tickets onto every monthly invoice.

In [ ]:
analysis = analysis.merge(
    contact_summary,
    on=["customer_id", "period"],
    how="left",
    validate="m:1",
)

- `on=["customer_id", "period"]` means both values must match. September contacts go to the September invoice, not to August.
- `how="left"` keeps invoices that had no CRM contact that month. Their counts will be missing for now; we will turn those missings into zeros later.
- `validate="m:1"`: after the aggregation, each `(customer_id, period)` pair appears at most once in `contact_summary`.

In [ ]:
show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_name",
        "period",
        "amount",
        "status",
        "n_interactions",
        "n_payment_issues",
        "n_reminders",
        "n_emails",
    ],
)

Still one row. The CRM has been reduced to numbers that belong on an invoice: Elena has a payment-issue email in September. The original sentences remain in `contacts` if we need them later. They do not belong on the analysis row we will plot.

## Step 5 — Attach the September API status

The API is not another grain of data. It is a **second source** for some of the same invoices. It only contains September 2026.

In [ ]:
api_invoices.loc[
    api_invoices["invoice_id"] == TARGET_INVOICE,
    ["invoice_id", "status", "amount"],
]

We rename `status` to `status_api` before merging. Otherwise pandas would create `status_x` and `status_y`, which are hard to read. The rename says clearly: this status comes from the API.

In [ ]:
api_status = api_invoices[["invoice_id", "status"]].rename(
    columns={"status": "status_api"}
)

analysis = analysis.merge(
    api_status,
    on="invoice_id",
    how="left",
    validate="m:1",
)

- The key is `invoice_id`.
- `how="left"` is essential. Most invoices in the panel are not from September. They have no API row. They must stay in the table, with `status_api` missing.
- `validate="1:1"` would be wrong. It would forbid any invoice from being missing on the right. That is not what we want. The API is a snapshot, not a complete history. Many historical invoices will not appear on the right. 

In [ ]:
show_invoice(
    analysis,
    ["invoice_id", "period", "status", "payment_status", "status_api", "amount"],
)

For this September invoice the three statuses agree: unpaid. For an August invoice, `status_api` would be empty. That missing value records “the snapshot does not cover this month”. It is not a broken join.

## The five attachments, in one view

| Step | Other table | Key copied from the invoice | What is added to the invoice row | Rows on the right for one invoice |
| --- | --- | --- | --- | --- |
| 1 | `customers` | `customer_id` | name, company, segment, country, signup date | 1 |
| 2 | `subscriptions` | `subscription_id` | plan, monthly amount, start date | 1 |
| 3 | `payments` | `invoice_id` | payment identifier, payment date, payment status | 1 |
| 4 | `contact_summary` | `customer_id` and `period` | counts of interactions that month | 1 after aggregation |
| 5 | `api_status` | `invoice_id` | `status_api` for September only | 1 or 0 |

The invoice identifier never changes. We only widen the row.

In [ ]:
analysis.shape

The number of rows should still equal the number of invoices. If it has grown, a one-to-many merge has leaked through.

## A few analysis columns

These columns are still preparation, not exploration. They are computed **from columns that the merges have already placed on the invoice**.

In [ ]:
analysis["days_to_due"] = (analysis["due_date"] - analysis["invoice_date"]).dt.days
analysis["days_to_pay"] = (analysis["payment_date"] - analysis["invoice_date"]).dt.days
analysis["paid_after_due"] = analysis["payment_date"].gt(analysis["due_date"])
analysis["is_outstanding"] = analysis["payment_status"].isin(["unpaid", "overdue"])
analysis["tenure_months"] = (
    (analysis["invoice_date"].dt.year - analysis["signup_date"].dt.year) * 12
    + (analysis["invoice_date"].dt.month - analysis["signup_date"].dt.month)
)

for column in [
    "n_interactions",
    "n_reminders",
    "n_complaints",
    "n_payment_issues",
    "n_emails",
]:
    analysis[column] = analysis[column].fillna(0).astype(int)

analysis.loc[analysis["payment_date"].isna(), "paid_after_due"] = False

- `days_to_pay` uses `payment_date`, which arrived in step 3. It is missing when there is no payment. That missingness **is** information.
- `tenure_months` uses `signup_date`, which arrived in step 1, and `invoice_date`, which was already on the invoice.
- Interaction counts that stayed missing after step 4 mean “no CRM contact that month”. We replace them by 0. A missing count would be the wrong story: no ticket is not “unknown”.

In [ ]:
show_invoice(
    analysis,
    [
        "invoice_id",
        "customer_name",
        "subscription_type",
        "amount",
        "payment_status",
        "status_api",
        "n_payment_issues",
        "is_outstanding",
        "tenure_months",
    ],
)

This is the analysis observation we wanted: one invoice, with customer, plan, payment, CRM and API information on the **same** row.

## Check that the joins are trustworthy

In [ ]:
print("Duplicate invoice ids:", analysis["invoice_id"].duplicated().sum())
print("Rows without a customer name:", analysis["customer_name"].isna().sum())
print("API status available:", analysis["status_api"].notna().sum())
print("Number of invoices:", len(invoices))
print("Number of analysis rows:", len(analysis))

If the last two numbers differ, an invoice was duplicated or dropped. If a customer name is missing, step 1 failed for that `customer_id`.

In [ ]:
september = analysis[analysis["period"] == "2026-09"]
(
    september["status"]
    .eq(september["status_api"])
    .value_counts(dropna=False)
)

For September, the workbook and the API should match. If they did not, we would have a **source conflict** to resolve before plotting.

A few customer segments are missing by construction. Later EDA will have to decide whether to drop those rows, recode them, or keep a missing category.

In [ ]:
analysis["customer_segment"].value_counts(dropna=False)

## Save the analysis table

In [ ]:
analysis = analysis.sort_values(["invoice_date", "customer_id"]).reset_index(drop=True)
analysis.to_csv("data/processed/harborlink_invoices.csv", index=False)
analysis.shape

`data/processed/harborlink_invoices.csv` is the table the next sessions will explore. The same pipeline is implemented in `python/build_analysis_frame.py` if you need to rebuild it without re-running every cell.

## Document the variables

Column names such as `n_payment_issues` or `status_api` are short for Python. Their meaning was not invented at the end of the join. Three commands carry the sentences from construction to Excel:

1. **Collect** — `write_source_description_files` (called by `python/generate_analysis_data.py`) writes one `*_variables.csv` next to each extract.
2. **Forward** — `read_descriptions` loads a companion; `attach_descriptions` copies selected rows onto the invoice catalog, with the same `rename` / `keep` / `skip` as the data joins.
3. **Write Excel** — `write_variable_workbooks(data, descriptions, dictionary_path, summary_path)` takes **any** data file and **any** description file and writes a dictionary workbook and a summary workbook.

The packaged forwarding step is `forward_invoice_descriptions` in `python/harborlink_catalog.py`. `python/build_analysis_frame.py` calls it when it rebuilds `harborlink_invoices.csv`, so the merged descriptions travel with the merged data even if you skip these cells.

### Where the sentences live

Open any companion file. The customer extract is typical: one row per column of `customers.csv`.

In [ ]:
from python.data_dictionary import read_descriptions, attach_descriptions, write_variable_workbooks

read_descriptions("data/raw/analysis/customers_variables.csv")

`read_descriptions` accepts a path (CSV, Excel or JSON) or a DataFrame. It only requires the three columns `variable`, `type`, `definition`.

### Transport: the same steps as the merges

`attach_descriptions` is the metadata analogue of `merge`. It starts from the invoice catalog and **adds** descriptions coming from the other sources. It does not look up rows; it looks up **column names**. `rename` is the same mapping we used on the data (`status` → `payment_status` in the SQL, `status` → `status_api` for the API). `keep` copies only the columns that the join actually attached. `skip` avoids writing `customer_id` twice.

The CRM counts (`n_interactions`, …) and the analysis columns (`days_to_due`, …) never existed in a source file. Their descriptions are attached at the moment those columns are created, just as the numbers themselves are.

In [ ]:
from python.harborlink_catalog import (
    analysis_column_descriptions,
    crm_count_descriptions,
)

descriptions = read_descriptions("data/raw/analysis/invoices_variables.csv")
descriptions = attach_descriptions(
    descriptions,
    "data/raw/analysis/customers_variables.csv",
    skip=["customer_id"],
)
descriptions = attach_descriptions(
    descriptions,
    "data/raw/analysis/subscriptions_variables.csv",
    keep=["subscription_type", "monthly_amount", "start_date"],
)
descriptions = attach_descriptions(
    descriptions,
    "data/raw/analysis/payments_variables.csv",
    rename={"status": "payment_status"},
    keep=["payment_id", "payment_date", "payment_status"],
)
descriptions = attach_descriptions(descriptions, crm_count_descriptions())
descriptions = attach_descriptions(
    descriptions,
    "data/raw/analysis/api_invoices_variables.csv",
    rename={"status": "status_api"},
    keep=["status_api"],
)
descriptions = attach_descriptions(descriptions, analysis_column_descriptions())
descriptions.to_csv("data/processed/harborlink_invoices_variables.csv", index=False)
descriptions

The CSV `harborlink_invoices_variables.csv` is the dictionary of the **merged** table, still as plain text, before any Excel formatting.

### One general function: two files in, two Excel files out

`write_variable_workbooks` is not a HarborLink helper. It gathers **two files** — a data set and a table of variable descriptions — and writes **two Excel files** — a dictionary and a column summary.

```
write_variable_workbooks(data, descriptions, dictionary_path, summary_path)
```

- `data`: a DataFrame, or a path to a CSV / Excel / JSON file;
- `descriptions`: a DataFrame, or a path to a CSV / Excel / JSON file with columns `variable`, `type`, `definition`;
- `dictionary_path`: where to write the glossary workbook;
- `summary_path`: where to write the summary workbook (hover notes, min / max / mean, histograms).

Here the two files are the merged invoice table and the forwarded descriptions we just saved:

In [ ]:
dictionary_path, summary_path = write_variable_workbooks(
    "data/processed/harborlink_invoices.csv",
    "data/processed/harborlink_invoices_variables.csv",
    "data/processed/harborlink_invoices_dictionary.xlsx",
    "data/processed/harborlink_invoices_summary.xlsx",
)
print(dictionary_path)
print(summary_path)

The same function on **two other files** — the customer extract and its own description CSV. Nothing in the call names invoices:

In [ ]:
write_variable_workbooks(
    "data/raw/analysis/customers.csv",
    "data/raw/analysis/customers_variables.csv",
    "data/processed/customers_dictionary.xlsx",
    "data/processed/customers_summary.xlsx",
)

If you introduce another pair (any rectangular data file plus a `*_variables.csv` with those three columns), pass those two paths and two output paths. The function does not change.

`harborlink_invoices_dictionary.xlsx` lists each merged column with its forwarded definition. `harborlink_invoices_summary.xlsx` is a portrait of the columns (not of the customers): hover a name in the first row to read the definition; below it, min / max / mean / median / standard deviation when the variable is numeric, then a histogram.

## Towards visualisation

We now have a rectangular, invoice-level HarborLink table: several thousand observations, with customer attributes, payment outcomes and CRM activity sitting on the same row **because each source was looked up with an explicit key**.

**The next session** starts from `data/processed/harborlink_invoices.csv` and uses **matplotlib** and **seaborn** to describe distributions, compare segments and plans, and look at late payments over time. That work is in `lessons/02d-eda-visualisation.qmd`.

The path remains:

**Heterogeneous sources → one lookup per source, on a named key → one DataFrame → exploratory analysis and visualisation → statistical and machine-learning analysis**